# Day 4

## Tokenizing with code

In [ ]:
# OpenAI creator of GPT
# import tiktoken
# encoding = tiktoken.encoding_for_model("gpt-4.1-mini")   #gpt comes form openai and llama from meta
# tokens = encoding.encode("Hi my name is Shweta and I like banoffee pie, she is great. She is learning")


# *******************************************



# Meta uploads their models to Hugging Face, and meta is creator of llama
from transformers import AutoTokenizer

# Load the Llama 3.2 tokenizer (you may need to log in via huggingface-cli if it's gated)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

text = "Hi my name is Shweta and I like banoffee pie, she is great. She is learning"

# Convert content into tokens
tokens = tokenizer.encode(text)

print(tokens)

[128000, 13347, 856, 836, 374, 1443, 86, 1955, 323, 358, 1093, 9120, 21869, 4447, 11, 1364, 374, 2294, 13, 3005, 374, 6975]


In [4]:
tokens

[128000,
 13347,
 856,
 836,
 374,
 1443,
 86,
 1955,
 323,
 358,
 1093,
 9120,
 21869,
 4447,
 11,
 1364,
 374,
 2294,
 13,
 3005,
 374,
 6975]

In [6]:
for token_id in tokens:
    token_text = tokenizer.decode([token_id])
    print(f"{token_id} = {token_text}")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


128000 = <|begin_of_text|>
13347 = Hi
856 =  my
836 =  name
374 =  is
1443 =  Sh
86 = w
1955 = eta
323 =  and
358 =  I
1093 =  like
9120 =  ban
21869 = offee
4447 =  pie
11 = ,
1364 =  she
374 =  is
2294 =  great
13 = .
3005 =  She
374 =  is
6975 =  learning


In [15]:
encoding.decode([326])

' and'

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [7]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

In [11]:
from openai import OpenAI

openai = OpenAI(base_url="http://localhost:11434/v1",api_key="ollama")

### A message to OpenAI is a list of dicts

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"}
    ]

In [ ]:
response = openai.chat.completions.create(model="llama3.2:1b", messages=messages)
response.choices[0].message.content

"Hello Ed! How's it going? What can I help you with today?"

### OK let's now ask a follow-up question

In [13]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [14]:
response = openai.chat.completions.create(model="llama3.2:1b", messages=messages)
response.choices[0].message.content

"I don't have any information about your identity, including your name. This conversation just started, and I'm here to help with any questions or topics you'd like to discuss. Is there something specific on your mind that you'd like to talk about, or do you want to get to know each other a bit better?"

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [17]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [18]:
response = openai.chat.completions.create(model="llama3.2:1b", messages=messages)
response.choices[0].message.content

'Your name is Ed, right?'

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

